# 02 · HBase – Wide-Column Modeling, Loading & Queries
**Part 1.2 of the Project.**

**Prerequisite:** HBase is running with its Thrift server (`docker compose up -d hbase`) and notebook 00 has generated the data.

## HBase in 60 seconds
* A table is a **sorted map**: `row key → column family:qualifier → value (versioned)`.
* Rows are stored **sorted by row key**, so the *only* fast access paths are **get by key**, **scan a key range** and **scan a key prefix**. → *The row key IS the schema.*
* **Column families** are physical storage units (kept few). **Qualifiers** are free: any row can have different columns (sparse, dynamic).
* Writes are very fast and scale linearly by adding region servers; there are no joins, no secondary indexes.

## What we store in HBase and why (vs. MongoDB)
| Data | Goes to | Reason |
|---|---|---|
| **Session event stream** (millions of rows, append-only, read by *user + time*) | **HBase** `user_sessions` | Key-range scans per user, newest-first; sparse variable columns (page views, cart lines); scales by region |
| **Product metrics per day** (counters, time series) | **HBase** `product_daily_metrics` | `product#date` key → a product's trend is one short scan; counters are atomic increments |
| Transactions, products, users, categories | **MongoDB** | Rich, ad-hoc aggregation (`$group`, `$lookup`, `$bucket`), nested documents, flexible secondary indexes |

>  *MongoDB answers "what / why / how much" questions across many fields (flexible analytics). HBase answers "give me everything for this key (range) – fast, at huge volume" (operational / time-series access).*

## Step 1 · Row-key design (the most important decision)

### Table 1 – `user_sessions`
```
row key  =  <user_id> # <reverse_timestamp> # <session_id>
            user_000042#8213876543210#sess_a7b3c9d8e2
reverse_timestamp = 9_999_999_999_999 − epoch_milliseconds   (13 digits, zero-padded)
```
| Design question | Decision | Why |
|---|---|---|
| Why `user_id` first? | Query pattern is *"sessions of user X"* | All of a user's rows are **adjacent** → a prefix scan reads one small region of disk |
| Why *reverse* timestamp? | HBase sorts ascending only | Newest session = smallest key → **"last N sessions" = scan with `limit=N`** (no sorting, no full scan) |
| Why zero-padded fixed width? | Keys sort as *bytes* | Without padding `"9" > "10"` and the order breaks |
| Why `session_id` suffix? | Two sessions can start in the same millisecond | Guarantees a unique key |
| Hot-spotting? | `user_000001…` is sequential, but *writes are spread by time of arrival, not by user id*; reads are per-user | For a production table with sequentially-registered users we would **salt** (`hash(user_id)%N` prefix) or pre-split regions; costs us ordered scans across users, which we never need |

| Column family | Qualifiers | Content |
|---|---|---|
| `meta` | session_id, user_id, start, end, duration, referrer, status, n_views | small, always read together |
| `device` | type, os, browser | |
| `geo` | city, state, country, ip | |
| `pv` | `000`, `001`, `002`, … | **one column per page view** (variable count per session). value = `timestamp\|page_type\|product_id\|category_id\|view_duration` |
| `cart` | `prod_00123`, … | **one column per cart line** – *the product id IS the qualifier* (dynamic, sparse). In MongoDB this shape was an anti-pattern; in HBase it is natural |

Few families (5), short names, `VERSIONS=1` (sessions are immutable events → no history wanted), `BLOOMFILTER=ROW` (fast point `get`).
Column families let a query read **only what it needs**: e.g. engagement analysis reads just `meta`, never the heavy `pv` family.

### Table 2 – `product_daily_metrics`
```
row key = <product_id>#<yyyymmdd>        prod_00123#20261015
family m : views | cart_adds | purchases | units | revenue
```
A product's whole trend = one **range scan**; "views today" can be maintained with atomic **counter increments** (stream-style updates).
*(Alternative key `yyyymmdd#product_id` would make "top products of a day" a scan – we choose product-first because our query is the per-product trend. A second table with the other key is the usual HBase answer when both are needed: **one table per access pattern**.)*

## Step 2 · Connect to HBASE

In [1]:
import sys; sys.path.append("../src")
from config import *
import json, glob, random, time, inspect
from collections import defaultdict
import pandas as pd
import happybase
import hbase_schema as HS

def connect():
    return happybase.Connection(HBASE_HOST, HBASE_PORT, timeout=60000)   # timeout in ms

try:
    conn = connect(); print("Connected. Existing tables:", conn.tables())
except Exception as e:
    raise SystemExit(f"HBase Thrift server not reachable on {HBASE_HOST}:{HBASE_PORT} -> docker compose up -d hbase  ({e})")
pd.set_option("display.max_columns", 30, "display.width", 200)

Connected. Existing tables: []


## Step 3 · Create the tables

In [2]:
RECREATE = True    # True = drop & recreate (safe to re-run the notebook)

def create_tables(conn, recreate=RECREATE):
    existing = {t.decode() for t in conn.tables()}
    specs = {
        HS.USER_SESSIONS:   {f: dict(max_versions=1, bloom_filter_type="ROW") for f in HS.USER_SESSIONS_FAMILIES},
        HS.PRODUCT_METRICS: {"m": dict(max_versions=1, bloom_filter_type="ROW")},
    }
    for name, fams in specs.items():
        if name in existing and recreate:
            conn.delete_table(name, disable=True); existing.discard(name); print("dropped", name)
        if name not in existing:
            conn.create_table(name, fams); print("created", name, "->", list(fams))

create_tables(conn)
sessions_tbl = conn.table(HS.USER_SESSIONS)
metrics_tbl  = conn.table(HS.PRODUCT_METRICS)
print(conn.tables())

created user_sessions -> ['meta', 'device', 'geo', 'pv', 'cart']
created product_daily_metrics -> ['m']
[b'product_daily_metrics', b'user_sessions']


> **Same thing in the HBase shell** (for the report; full file in `hbase/hbase_shell_commands.txt`):
> ```
> create 'user_sessions', {NAME=>'meta',VERSIONS=>1,BLOOMFILTER=>'ROW'}, {NAME=>'device',VERSIONS=>1},
>   {NAME=>'geo',VERSIONS=>1}, {NAME=>'pv',VERSIONS=>1}, {NAME=>'cart',VERSIONS=>1},
>   {SPLITS=>['user_002000','user_004000','user_006000','user_008000']}
> create 'product_daily_metrics', {NAME=>'m',VERSIONS=>1,BLOOMFILTER=>'ROW'}
> ```
> `SPLITS` pre-splits the table into 5 regions so a bulk load is spread over several region servers from the first second (not possible through the happybase API, which is why the Python version omits it).

## Step 4 · The (de)serialisation code
Read it once – it implements the row-key and column design above and is unit-tested (`pytest`).

In [3]:
for fn in (HS.user_session_rowkey, HS.user_time_range_bounds, HS.session_to_cells):
    print(inspect.getsource(fn))

def user_session_rowkey(user_id: str, start_iso: str, session_id: str) -> bytes:
    rev = MAX_TS_MS - to_epoch_ms(start_iso)
    return f"{user_id}#{rev:013d}#{session_id}".encode()

def user_time_range_bounds(user_id: str, start_iso: str, end_iso: str):
    """(row_start, row_stop) covering sessions that started in [start_iso, end_iso].
    Newest sessions have SMALLER reverse timestamps, hence the swap."""
    rev_from = MAX_TS_MS - to_epoch_ms(end_iso)
    rev_to = MAX_TS_MS - to_epoch_ms(start_iso) + 1      # stop row is exclusive
    return (f"{user_id}#{rev_from:013d}".encode(), f"{user_id}#{rev_to:013d}".encode())

def session_to_cells(s: dict) -> dict:
    """One raw session dict -> {b'family:qualifier': b'value'} ready for table.put()."""
    cells = {
        b"meta:session_id": _s(s["session_id"]),
        b"meta:user_id": _s(s["user_id"]),
        b"meta:start": _s(s["start_time"]),
        b"meta:end": _s(s["end_time"]),
        b"meta:duration": _s(s["duration_seconds"])

## Step 5 · Load session data
We stream the session files and write with `table.batch(batch_size=…)` – the client buffers puts and sends them in one network round trip. Each chunk is retried (with reconnect) up to 3 times, so a Thrift hiccup doesn't ruin a long load. Re-running is **idempotent** (same row key → same row, `VERSIONS=1`).

`MAX_SESSIONS` lets you load a **representative subset** first (the assignment asks for "a significant representative subset"). Start with `50_000`, raise it if time allows.

In [4]:
MAX_SESSIONS = 200_000     # None = all sessions in data/raw ; 50_000 is a good first run
CHUNK = 2000

def load_chunk(rows, attempts=3):
    global conn, sessions_tbl
    for a in range(1, attempts + 1):
        try:
            with sessions_tbl.batch(batch_size=1000) as b:
                for key, cells in rows:
                    b.put(key, cells)
            return
        except Exception as e:
            print(f"  ! chunk failed (attempt {a}/{attempts}): {e}")
            time.sleep(2 * a); conn = connect(); sessions_tbl = conn.table(HS.USER_SESSIONS)
    raise RuntimeError("chunk failed 3 times - check that HBase is healthy (docker logs hbase)")

loaded, t0 = 0, time.perf_counter()
for f in sorted(glob.glob(str(RAW / "sessions_*.json"))):
    part = json.load(open(f))
    buf = []
    for s in part:
        if MAX_SESSIONS is not None and loaded + len(buf) >= MAX_SESSIONS: break
        buf.append((HS.user_session_rowkey(s["user_id"], s["start_time"], s["session_id"]), HS.session_to_cells(s)))
        if len(buf) >= CHUNK:
            load_chunk(buf); loaded += len(buf); buf = []
            if loaded % 20000 == 0: print(f"  {loaded:,} sessions  ({loaded / (time.perf_counter() - t0):,.0f} rows/s)")
    if buf: load_chunk(buf); loaded += len(buf)
    del part
    if MAX_SESSIONS is not None and loaded >= MAX_SESSIONS: break
print(f"Loaded {loaded:,} sessions in {time.perf_counter() - t0:,.0f}s  ({loaded / (time.perf_counter() - t0):,.0f} rows/s)")

  20,000 sessions  (899 rows/s)
  40,000 sessions  (1,201 rows/s)
  60,000 sessions  (1,378 rows/s)
  80,000 sessions  (1,469 rows/s)
  100,000 sessions  (1,543 rows/s)
  120,000 sessions  (1,468 rows/s)
  140,000 sessions  (1,509 rows/s)
  160,000 sessions  (1,526 rows/s)
  180,000 sessions  (1,536 rows/s)
  200,000 sessions  (1,548 rows/s)
Loaded 200,000 sessions in 130s  (1,538 rows/s)


In [5]:
# Verify: count rows with a key-only scan (cheap - no cell data is transferred)
with timer("row count"):
    n = sum(1 for _ in sessions_tbl.scan(filter="FirstKeyOnlyFilter()", batch_size=5000))
print(f"user_sessions rows: {n:,}")
assert n == loaded, "row count differs from the number of sessions loaded"

[timer] row count: 10.80s
user_sessions rows: 200,000


## Step 6 · Load product daily metrics
Computed once from sessions + transactions (this is what a nightly Spark/stream job would write in production), then written with one `put` per `(product, day)` row.
* `views`     – product_detail page views
* `cart_adds` – sessions where the product was in the cart
* `purchases` – orders containing it; `units`, `revenue` – from transactions

In [6]:
metrics = defaultdict(lambda: {"views": 0, "cart_adds": 0, "purchases": 0, "units": 0, "revenue": 0.0})
day_of = lambda iso: iso[:10].replace("-", "")

n_s = 0
for f in sorted(glob.glob(str(RAW / "sessions_*.json"))):
    for s in json.load(open(f)):
        if MAX_SESSIONS is not None and n_s >= MAX_SESSIONS: break
        n_s += 1
        d = day_of(s["start_time"])
        for pv in s["page_views"]:
            if pv["page_type"] == "product_detail" and pv["product_id"]:
                metrics[(pv["product_id"], d)]["views"] += 1
        for pid in (s["cart_contents"] or {}):
            metrics[(pid, d)]["cart_adds"] += 1
for t in json.load(open(RAW / "transactions.json")):
    d = day_of(t["timestamp"])
    for it in t["items"]:
        m = metrics[(it["product_id"], d)]
        m["purchases"] += 1; m["units"] += it["quantity"]; m["revenue"] += it["subtotal"]
print(f"{len(metrics):,} (product, day) rows to write")

with timer("write product_daily_metrics"):
    with metrics_tbl.batch(batch_size=2000) as b:
        for (pid, d), m in metrics.items():
            b.put(HS.product_metric_rowkey(pid, d), {f"m:{k}".encode(): str(round(v, 2) if k == "revenue" else v).encode() for k, v in m.items()})

355,581 (product, day) rows to write
[timer] write product_daily_metrics: 36.73s


## Step 7 · Queries
These are the access patterns the schema was designed for. Every one is a **get, prefix scan or range scan – never a full-table scan**.

### Q1 – All sessions of one user (prefix scan)
*(This is the query the assignment explicitly asks for.)*

In [7]:
all_users = sorted({u["user_id"] for u in json.load(open(RAW / "users.json"))})
user = next(u for u in all_users if any(True for _ in sessions_tbl.scan(row_prefix=HS.user_prefix(u), limit=1)))
print("Chosen user:", user)

def user_sessions(user_id, limit=None, columns=None):
    return [HS.cells_to_session(k, d) for k, d in sessions_tbl.scan(row_prefix=HS.user_prefix(user_id), limit=limit, columns=columns)]

with timer("Q1 all sessions of the user"):
    rows = user_sessions(user)
print(f"{len(rows)} sessions")
display(pd.DataFrame([{k: r[k] for k in ("start", "duration", "referrer", "status")} | {"device": r["device"]["type"], "page_views": len(r["page_views"]), "cart_lines": len(r["cart"])} for r in rows]).head(10))

Chosen user: user_000000
[timer] Q1 all sessions of the user: 0.01s
13 sessions


,start,duration,referrer,status,device,page_views,cart_lines
0,2026-10-30T03:08:45,2795,email,abandoned,desktop,9,2
1,2026-10-22T18:13:37,1423,search_engine,abandoned,desktop,15,4
2,2026-10-09T13:02:30,100,direct,browsed,desktop,13,0
3,2026-10-04T18:48:20,1287,social,abandoned,tablet,10,1
4,2026-09-24T13:24:53,2216,search_engine,browsed,mobile,5,0
5,2026-09-19T09:25:00,2105,email,abandoned,tablet,16,1
6,2026-09-14T14:07:53,1949,direct,browsed,mobile,11,0
7,2026-09-11T02:50:09,2515,direct,abandoned,mobile,10,2
8,2026-08-25T19:07:38,3410,affiliate,browsed,mobile,11,0
9,2026-08-24T14:22:02,643,direct,converted,tablet,10,2


### Q2 – The user's 5 most recent sessions (thanks to the reverse timestamp: `limit=5`, no sorting)

In [8]:
with timer("Q2 last 5 sessions"):
    latest = user_sessions(user, limit=5)
print([r["start"] for r in latest], "<- newest first")
assert [r["start"] for r in latest] == sorted([r["start"] for r in rows], reverse=True)[:5]

[timer] Q2 last 5 sessions: 0.01s
['2026-10-30T03:08:45', '2026-10-22T18:13:37', '2026-10-09T13:02:30', '2026-10-04T18:48:20', '2026-09-24T13:24:53'] <- newest first


### Q3 – Sessions of that user in a time window (range scan with computed start/stop rows)

In [9]:
import datetime as dt
win_end = rows[0]["start"]                      # the user's newest session (full ISO string)
win_start = (dt.datetime.fromisoformat(win_end) - dt.timedelta(days=14)).isoformat()
r0, r1 = HS.user_time_range_bounds(user, win_start, win_end)
print("scan from", r0.decode(), "\n     to ", r1.decode())
with timer("Q3 time-window scan"):
    win = [HS.cells_to_session(k, d) for k, d in sessions_tbl.scan(row_start=r0, row_stop=r1)]
print(f"{len(win)} sessions between {win_start[:10]} and {win_end[:10]}")
assert all(HS.to_epoch_ms(win_start) <= HS.to_epoch_ms(r["start"]) <= HS.to_epoch_ms(win_end) for r in win)

scan from user_000000#8206670274999 
     to  user_000000#8207879875000
[timer] Q3 time-window scan: 0.02s
2 sessions between 2026-10-16 and 2026-10-30


### Q4 – Rebuild one full session journey from a single row (`get`)

In [10]:
key = latest[0]["row_key"].encode()
with timer("Q4 point get"):
    row = sessions_tbl.row(key)
sess = HS.cells_to_session(key, row)
print("Session", sess["session_id"], "|", sess["device"], "|", sess["referrer"], "|", sess["status"])
display(pd.DataFrame(sess["page_views"]))
print("Cart:", sess["cart"])

[timer] Q4 point get: 0.08s
Session sess_9dad2a50c8 | {'type': 'desktop', 'os': 'Android', 'browser': 'Chrome'} | email | abandoned


,timestamp,page_type,product_id,category_id,view_duration
0,2026-10-30T03:08:45,home,NaN,NaN,152
1,2026-10-30T03:11:17,search,NaN,NaN,97
2,2026-10-30T03:12:54,product_detail,prod_01184,cat_020,300
3,2026-10-30T03:17:54,category_listing,NaN,cat_003,37
4,2026-10-30T03:18:31,product_detail,prod_00117,cat_005,581
5,2026-10-30T03:28:12,home,NaN,NaN,226
6,2026-10-30T03:31:58,category_listing,NaN,cat_008,455
7,2026-10-30T03:39:33,product_detail,prod_00369,cat_006,144
8,2026-10-30T03:41:57,product_detail,prod_00313,cat_016,803


Cart: {'prod_00313': {'quantity': 3, 'price': 253.78}, 'prod_00369': {'quantity': 3, 'price': 439.15}}


### Q5 – Read only the columns you need (column-family pruning)
Engagement analysis needs `meta:duration` only – HBase never touches the heavy `pv` family.

In [11]:
with timer("Q5 columns=['meta:duration','meta:n_views'] for this user"):
    slim = list(sessions_tbl.scan(row_prefix=HS.user_prefix(user), columns=[b"meta:duration", b"meta:n_views"]))
print(slim[:2])

[timer] Q5 columns=['meta:duration','meta:n_views'] for this user: 0.06s
[(b'user_000000#8206670274999#sess_9dad2a50c8', {b'meta:duration': b'2795', b'meta:n_views': b'9'}), (b'user_000000#8207307182999#sess_2b03354d8e', {b'meta:duration': b'1423', b'meta:n_views': b'15'})]


### Q6 – A product's daily performance over a date range (range scan on `product_daily_metrics`)

In [12]:
views_by_product = defaultdict(int)
for (pid, _), m in metrics.items():
    views_by_product[pid] += m["views"]
top_pid = max(views_by_product, key=views_by_product.get)
r0, r1 = HS.product_range_bounds(top_pid, "20260101", "20261231")
with timer("Q6 product trend scan"):
    trend = pd.DataFrame([{"date": k.decode().split("#")[1], **{c.decode().split(":")[1]: float(v) for c, v in d.items()}}
                          for k, d in metrics_tbl.scan(row_start=r0, row_stop=r1)])
print("Most viewed product:", top_pid, "|", len(trend), "days")
display(trend.head(10))
trend.to_csv(TAB / "hbase_product_trend_sample.csv", index=False)

[timer] Q6 product trend scan: 0.02s
Most viewed product: prod_04857 | 81 days


,date,cart_adds,purchases,revenue,units,views
0,20260812,1.0,0.0,0.00,0.0,1.0
1,20260814,1.0,0.0,0.00,0.0,3.0
2,20260815,0.0,1.0,57.88,1.0,0.0
3,20260816,1.0,1.0,115.76,2.0,4.0
4,20260817,0.0,0.0,0.00,0.0,1.0
5,20260818,0.0,0.0,0.00,0.0,4.0
6,20260819,0.0,0.0,0.00,0.0,1.0
7,20260821,0.0,0.0,0.00,0.0,1.0
8,20260822,0.0,0.0,0.00,0.0,4.0
9,20260823,1.0,1.0,115.76,2.0,1.0


### Q7 – Atomic counters (how HBase supports *streaming* updates)
Each incoming page-view event would do one `counter_inc` on the product/day cell – atomic, no read-modify-write race, no lock. We demonstrate on a throw-away row.

In [13]:
demo = b"demo_product#20991231"
for _ in range(5):
    v = metrics_tbl.counter_inc(demo, b"m:views")
print("counter after 5 events:", v, "| current value:", metrics_tbl.counter_get(demo, b"m:views"))
metrics_tbl.delete(demo)

counter after 5 events: 5 | current value: 5


## Step 8 · Why the row key matters – measured
We fetch the sessions of **100 random users** two ways:
1. **Prefix scan** (what the schema was built for)
2. **Anti-pattern:** full-table scan, filtering `user_id` on the client

The ratio is the number to quote in the report.

In [14]:
sample_users = random.sample(all_users, 100)

t0 = time.perf_counter()
n1 = sum(len(list(sessions_tbl.scan(row_prefix=HS.user_prefix(u), columns=[b"meta:duration"]))) for u in sample_users)
t_prefix = time.perf_counter() - t0

wanted = {HS.user_prefix(u) for u in sample_users}
t0 = time.perf_counter()
n2 = sum(1 for k, _ in sessions_tbl.scan(columns=[b"meta:duration"], batch_size=5000) if k.split(b"#")[0] + b"#" in wanted)
t_full = time.perf_counter() - t0
assert n1 == n2
print(f"100 prefix scans: {t_prefix:6.2f}s  ({t_prefix / 100 * 1000:5.1f} ms per user)")
print(f"1 full scan     : {t_full:6.2f}s  (one pass, but it reads EVERY row, and must be repeated for every new question)")
print(f"-> a single user lookup via the key is ~{t_full / (t_prefix / 100):,.0f}x cheaper than scanning the table")

100 prefix scans:   1.45s  ( 14.5 ms per user)
1 full scan     :   4.84s  (one pass, but it reads EVERY row, and must be repeated for every new question)
-> a single user lookup via the key is ~333x cheaper than scanning the table


## Step 9 · Scalability notes 
* **Reads/writes scale with the number of regions** – HBase splits a table into regions by key range and spreads them over region servers; adding servers adds throughput. Our single Docker container is one region server; the *design* is unchanged on a 100-node cluster.
* **Pre-split + (optionally) salt** the `user_sessions` table before bulk-loading to avoid all writes hitting the first region.
* **Per-user reads touch a tiny, contiguous key range** → latency stays in milliseconds even at billions of rows. This is *independent of table size*, unlike a MongoDB query without a matching index.
* **Cost:** no ad-hoc queries – a new question ("sessions by referrer") needs either a full scan (→ do it in **Spark**, notebook 03/05) or another table with a different key.

###  What you should have after this notebook
* Tables `user_sessions` (≈ 200 k rows) and `product_daily_metrics` in HBase.
* Output of Q1–Q7 and the prefix-vs-full-scan timing.
* `outputs/tables/hbase_product_trend_sample.csv`.

➡ Next: `03_spark_batch_processing.ipynb`.